In [33]:
import requests 
from bs4 import BeautifulSoup
import time
import pandas as pd

box_scores_url = "https://www.sports-reference.com/cbb/boxscores/index.cgi?month={}&day={}&year={}"

def get_day_games_data(month: int, day: int, year: int) -> pd.DataFrame:
    """Get all games data for a given day."""
    date = f"{year}-{month:02d}-{day:02d}"
    r = requests.get(box_scores_url.format(month, day, year))
    if r.status_code != 200:
        print(f"Rate limited. Sleeping for {r.headers['Retry-After']} seconds")
        time.sleep(r.headers["Retry-After"])
        r = requests.get(box_scores_url.format(month, day, year))
    soup = BeautifulSoup(r.text, "html.parser")
    summaries = soup.find("div", class_="game_summaries")
    if not summaries:
        print("No games today ...")
        return pd.DataFrame()
    
    men_games = summaries.find_all("div", class_="game_summary nohover gender-m")
    men_team_data_rows = []

    for game in men_games:
        teams = game.find_all("tr", class_=["winner", "loser"])
        for team in teams:
            location = "Home" if teams.index(team) == 1 else "Away"
            team_name = team.find("a").text
            pollrank = team.find("span", class_="pollrank")
            if pollrank:
                rank = pollrank.text.replace("(", "").replace(")", "")
            else: 
                rank = "NR"
            team_data_row = {
                "GameID": f"{date}-{teams[0].find('a').text}-vs-{teams[1].find('a').text}-m",
                "Date": date,
                "Team": team_name,
                "Opponent": [i for i in teams if i != team][0].find("a").text,
                "Location": location,
                "Rank": rank,
                "Opponent Rank" :  [i for i in teams if i != team][0].find("span", class_="pollrank").text.replace("(", "").replace(")", "") if [i for i in teams if i != team][0].find("span", class_="pollrank") else "NR"
            }
            men_team_data_rows.append(team_data_row)

    men_teams_df = pd.DataFrame(men_team_data_rows)

    return men_teams_df

In [28]:
def get_season_data(dates: list[tuple[int, int, int]], throttle: int = 15, throttle_time: int = 60):
    """Get all season data for given"""
    teams_df = pd.DataFrame()
    pages_visited = 0
    start_time = time.time()
    for month, day, year in dates:
        men_teams_df = get_day_games_data(month, day, year)
        teams_df = pd.concat([teams_df, men_teams_df], ignore_index=True)
        print("Done with date:", f"{year}-{month:02d}-{day:02d}")
        pages_visited += 1
        if pages_visited == throttle: 
            print(f"Sleeping for {throttle_time - (time.time() - start_time):.2f} seconds ...")
            time.sleep(throttle_time - (time.time() - start_time))
            pages_visited = 0
            start_time = time.time()
    return teams_df

In [29]:
def get_dates(start, end):
    """Generate a list of dates between start and end."""
    from datetime import datetime, timedelta
    start_date = datetime.strptime(start, "%Y-%m-%d")
    end_date = datetime.strptime(end, "%Y-%m-%d")
    delta = end_date - start_date
    dates = []
    for i in range(delta.days + 1):
        day = start_date + timedelta(days=i)
        dates.append((day.month, day.day, day.year))
    return dates

In [34]:
dates = get_dates("2025-11-03", "2026-03-08")
schedule = get_season_data(dates, 15, 60)

Done with date: 2025-11-03
Done with date: 2025-11-04
Done with date: 2025-11-05
Done with date: 2025-11-06
Done with date: 2025-11-07
Done with date: 2025-11-08
Done with date: 2025-11-09
Done with date: 2025-11-10
Done with date: 2025-11-11
Done with date: 2025-11-12
Done with date: 2025-11-13
Done with date: 2025-11-14
Done with date: 2025-11-15
Done with date: 2025-11-16
Done with date: 2025-11-17
Sleeping for 55.11 seconds ...
Done with date: 2025-11-18
Done with date: 2025-11-19
Done with date: 2025-11-20
Done with date: 2025-11-21
Done with date: 2025-11-22
Done with date: 2025-11-23
Done with date: 2025-11-24
Done with date: 2025-11-25
Done with date: 2025-11-26
Done with date: 2025-11-27
Done with date: 2025-11-28
Done with date: 2025-11-29
Done with date: 2025-11-30
Done with date: 2025-12-01
Done with date: 2025-12-02
Sleeping for 55.65 seconds ...
Done with date: 2025-12-03
Done with date: 2025-12-04
Done with date: 2025-12-05
Done with date: 2025-12-06
Done with date: 2025

In [36]:
schedule.to_csv("schedule.csv")